# Phase 2: full-data study

Thirty-five runs at 1,044 training images with seed 42: the four primary cells
A to D, the two capacity controls E and F, and the contextual linear probe,
each across five folds. After the runs, the analysis section computes C1, C2,
and C3 with the 10,000-replicate paired group bootstrap and the Bonferroni
adjustment across the C1-C4 family.

Training needs the GPU pod. RUNBOOK.md: stop or reduce the pod after training
and run the analysis section on a CPU machine; the manifest skips finished runs.


In [1]:
%run definitions.ipynb
require_pythonhashseed()
PH2 = PROJECT_ROOT / "phase2" / "outputs"
PH2.mkdir(parents=True, exist_ok=True)
ENV = environment_record()
SPLITS_CTX = load_split_tables()
COHORT, FOLDS, INNER, LABELS, GROUPS = SPLITS_CTX
print("phase 2 ready |", len(phase2_run_specs()), "registered runs")


required packages already match the pins.
torch 2.8.0+cu128 present (training phases need >= 2.8.0).
torchvision 0.23.0+cu128 present (training phases need >= 0.23.0).


python 3.12.3 | numpy 2.2.6 | pandas 2.3.3 | pillow 12.0.0 | opencv 4.12.0 | torch available: True


phase 2 ready | 35 registered runs


In [2]:
P2_STATUS = execute_pending_runs(PH2, phase2_run_specs(), SPLITS_CTX, "phase2.ipynb")


[1/8] p2_F_f3_s42_n1044


[2/8] p2_F_f4_s42_n1044


[3/8] p2_F_f5_s42_n1044


[4/8] p2_probe_f1_s42_n1044


[5/8] p2_probe_f2_s42_n1044


[6/8] p2_probe_f3_s42_n1044


[7/8] p2_probe_f4_s42_n1044


[8/8] p2_probe_f5_s42_n1044


In [3]:
# Per-run secondary metrics from the exported predictions and thresholds.
if P2_STATUS == "complete":
    rows = []
    for spec in phase2_run_specs():
        run_dir = PH2 / "runs" / spec["run_id"]
        meta = json.load(open(run_dir / "meta.json"))
        thresholds = json.load(open(run_dir / "thresholds.json"))
        test = load_test_predictions(PH2, spec["run_id"])
        mac, per_class = macro_auc(test["labels"], test["probs"])
        row = {"run_id": spec["run_id"], "cell": spec["cell"], "fold": spec["fold"],
               "test_macro_auc": mac,
               **{f"auc_{t}": v for t, v in zip(TARGETS, per_class)},
               **thresholded_metrics(test["labels"], test["probs"], thresholds),
               "warmup_epochs_run": meta["warmup_epochs_run"],
               "finetune_best_epoch": meta["finetune_best_epoch"],
               "trainable_parameters": meta["trainable_parameters"],
               "wall_seconds": meta["wall_seconds"],
               "peak_gpu_gb": round(meta["peak_gpu_bytes"] / 1e9, 2)}
        row.pop("per_class_f1")
        rows.append(row)
    run_metrics = pd.DataFrame(rows)
    run_metrics.to_csv(PH2 / "run_metrics.csv", index=False)
    cell_summary = run_metrics.groupby("cell")["test_macro_auc"].agg(["mean", "std"])
    print(cell_summary.to_string())
else:
    print("PENDING: metrics wait for the training runs.")


           mean       std
cell                     
A      0.908876  0.008417
B      0.895227  0.008785
C      0.913123  0.010750
D      0.906049  0.004946
E      0.909294  0.010566
F      0.907334  0.011663
probe  0.881992  0.013414


In [4]:
# Bootstrap preparation: load predictions, build the group structures, and
# define the per-replicate computation that the parity check and the full run
# share. Indices come from draw_replicates only.
BOOT_READY = False
if P2_STATUS == "complete":
    cells = ["A", "B", "C", "D", "E", "F"]
    preds = {(cell, fold): load_test_predictions(PH2, f"p2_{cell}_f{fold}_s42_n1044")
             for cell in cells for fold in range(1, N_FOLDS + 1)}
    fold_ids = {f: preds[("A", f)]["image_ids"] for f in range(1, N_FOLDS + 1)}
    fold_labels = {f: preds[("A", f)]["labels"] for f in range(1, N_FOLDS + 1)}
    fold_groups = {f: fold_group_index_lists(fold_ids[f], GROUPS) for f in fold_ids}
    point_auc = {c: {f: macro_auc(preds[(c, f)]["labels"], preds[(c, f)]["probs"])[0]
                     for f in range(1, N_FOLDS + 1)} for c in cells}

    def make_phase2_replicate(reps_local, masks_local):
        def _replicate(r):
            fold_auc = {cll: np.zeros(N_FOLDS) for cll in cells}
            included_counts = []
            for f in range(1, N_FOLDS + 1):
                idx = reps_local[r][f]
                included = [k for k in range(len(TARGETS)) if masks_local[r][f][k]]
                included_counts.append(len(included))
                for cll in cells:
                    p = preds[(cll, f)]
                    fold_auc[cll][f - 1] = macro_auc(p["labels"][idx], p["probs"][idx],
                                                     included)[0]
            av, bv, cv, dv, ev, fv = (fold_auc[x] for x in cells)
            return (float(np.mean(0.5 * ((cv - av) + (dv - bv)))),
                    float(np.mean(0.5 * ((bv - av) + (dv - cv)))),
                    float(np.mean((dv - cv) - (bv - av))),
                    float(np.mean(0.5 * ((ev - av) + (fv - bv)))),
                    float(np.mean(0.5 * ((cv - ev) + (dv - fv)))),
                    tuple(included_counts))
        return _replicate

    BOOT_READY = True
    print("bootstrap preparation done")
else:
    print("PENDING: bootstrap preparation waits for the training runs.")


bootstrap preparation done


In [5]:
# Parity check: 500 replicates through the serial and the forked path must be
# bit-identical before the full bootstrap runs.
if BOOT_READY:
    reps500, masks500, _ = draw_replicates(fold_groups, 500, BOOTSTRAP_SEED, fold_labels)
    check_fn = make_phase2_replicate(reps500, masks500)
    t0 = time.time()
    serial_out = run_replicates(check_fn, 500, parallel=False)
    t1 = time.time()
    forked_out = run_replicates(check_fn, 500)
    t2 = time.time()
    if serial_out != forked_out:
        raise PhaseStop("parallel bootstrap output differs from the serial output.")
    print(f"parity check passed: 500 replicates bit-identical "
          f"(serial {t1 - t0:.1f}s, parallel {t2 - t1:.1f}s, {os.cpu_count()} cpus)")
else:
    print("PENDING")


parity check passed: 500 replicates bit-identical (serial 6.0s, parallel 6.2s, 255 cpus)


In [6]:
# C1-C3 with the paired group bootstrap, replicates in parallel across cores.
# BOOTSTRAP_REPLICATES=10000 produces the reported inference. Smaller values
# are for pipeline testing only.
if BOOT_READY:
    estimates = contrast_point_estimates(point_auc)
    a, b = (np.array([point_auc[x][f] for f in range(1, N_FOLDS + 1)]) for x in ["A", "B"])
    c, d, e, fq = (np.array([point_auc[x][f] for f in range(1, N_FOLDS + 1)])
                   for x in ["C", "D", "E", "F"])
    capacity = {"capacity_effect": float(np.mean(0.5 * ((e - a) + (fq - b)))),
                "mhc_versus_capacity": float(np.mean(0.5 * ((c - e) + (d - fq))))}

    print("drawing bootstrap replicates...")
    reps, masks, discarded = draw_replicates(fold_groups, BOOTSTRAP_REPLICATES,
                                             BOOTSTRAP_SEED, fold_labels)
    rep_fn = make_phase2_replicate(reps, masks)
    t0 = time.time()
    results = run_replicates(rep_fn, len(reps))
    print(f"{len(results)} replicates computed in {time.time() - t0:.0f}s")
    boot = {"C1": [r[0] for r in results], "C2": [r[1] for r in results],
            "C3": [r[2] for r in results],
            "capacity_effect": [r[3] for r in results],
            "mhc_versus_capacity": [r[4] for r in results]}
    class_counts = [k for r in results for k in r[5]]

    confirmatory = [bootstrap_summary(k, estimates[k], boot[k]) for k in ["C1", "C2", "C3"]]
    exploratory = [bootstrap_summary(k, capacity[k], boot[k])
                   for k in ["capacity_effect", "mhc_versus_capacity"]]
    for row in exploratory:
        row["p_adjusted"] = None  # exploratory contrasts carry no family correction
    analysis = {
        "replicates": BOOTSTRAP_REPLICATES, "bootstrap_seed": BOOTSTRAP_SEED,
        "discarded_replicates": discarded,
        "mean_included_classes": float(np.mean(class_counts)),
        "fold_macro_auc": {cll: point_auc[cll] for cll in cells},
        "confirmatory_C1_C2_C3": confirmatory,
        "exploratory_capacity": exploratory,
        "pooled_oof_note": "pooled out-of-fold macro AUC is secondary",
    }
    pooled = {}
    for cll in cells:
        labels_all = np.concatenate([preds[(cll, f)]["labels"] for f in range(1, N_FOLDS + 1)])
        probs_all = np.concatenate([preds[(cll, f)]["probs"] for f in range(1, N_FOLDS + 1)])
        pooled[cll] = macro_auc(labels_all, probs_all)[0]
    analysis["pooled_oof_macro_auc"] = pooled
    write_json(PH2 / "analysis_c1_c3.json", analysis)
    print(pd.DataFrame(confirmatory).to_string(index=False))
    print(pd.DataFrame(exploratory).to_string(index=False))
else:
    print("PENDING: the C1-C3 analysis waits for the training runs.")


drawing bootstrap replicates...


10000 replicates computed in 21s
contrast  estimate    ci_low   ci_high    p_raw  p_adjusted  replicates
      C1  0.007534  0.001243  0.013579 0.002000    0.007999       10000
      C2 -0.010361 -0.016435 -0.004442 0.000100    0.000400       10000
      C3  0.006575 -0.005061  0.018294 0.163184    0.652735       10000
           contrast  estimate    ci_low  ci_high    p_raw p_adjusted  replicates
    capacity_effect  0.006263  0.000458 0.012079 0.006999       None       10000
mhc_versus_capacity  0.001271 -0.004128 0.006381 0.557544       None       10000


In [7]:
manifest = manifest_frame(PH2)
ok = (manifest["status"] == "success").sum() if len(manifest) else 0
if P2_STATUS == "complete" and (PH2 / "analysis_c1_c3.json").exists():
    required = {"checkpoint.pt", "history.csv", "predictions_inner.npz",
                "predictions_test.npz", "thresholds.json", "meta.json"}
    missing = [s["run_id"] for s in phase2_run_specs()
               if not required.issubset({p.name for p in (PH2 / "runs" / s["run_id"]).iterdir()})]
    if missing:
        raise PhaseStop(f"runs missing files: {missing}")
    print(f"35 successful runs, complete files, analysis written. PHASE 2 COMPLETE")
else:
    print(f"PHASE 2 PENDING: {ok} successful runs so far.")


35 successful runs, complete files, analysis written. PHASE 2 COMPLETE
